# 히스테리시스 + Hard 40% + 레버리지 상한 실험

노트북의 ±0.2 히스테리시스를 현재 Hard 40% 배분에만 적용하고, 최종 레버리지 상한 1.0·1.1·1.2·1.3을 비교합니다.

- `hysteresis_hard40_leverage_colab_bundle.zip`을 함께 업로드하세요.
- 상한 선택은 2017년까지의 자료만 사용하고, 2018–2026은 선택 후 검증합니다.
- 기존 무SJM 거시 확률, SLSQP 60%, 균형 L2 로지스틱, Robust VKOSPI는 유지됩니다.
- CPU 런타임에서 실행하며 GPU는 필요하지 않습니다.


## 1. 환경과 입력 번들 준비

In [ ]:
import sys, subprocess, zipfile, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "scipy", "scikit-learn", "openpyxl", "matplotlib"], check=True)

def safe_extract(zip_path: Path, destination: Path) -> None:
    destination = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
        archive.extractall(destination)

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    bundle = next((Path(name) for name in uploaded if name.endswith(".zip")), None)
    if bundle is None:
        raise FileNotFoundError("hysteresis_hard40_leverage_colab_bundle.zip을 업로드하세요.")
    safe_extract(bundle, Path("/content"))
    PROJECT_ROOT = Path("/content/RegimeDecisionTest")
else:
    PROJECT_ROOT = Path.cwd().resolve()

required = [
    "strategies/stage09_hysteresis/hysteresis_hard40_leverage_experiment.py",
    "strategies/stage06_vkospi/balanced_logistic_no_sjm_strategy.py",
    "strategies/core/regime_research.py",
    "cache/market_daily.csv",
    "raw_data/compass.db",
    "raw_data/VKOSPIData.csv",
    "results/openassetpricing_composites.csv",
    "results/vkospi_robust_dynamic_validation.json",
    "results/balanced_logistic_no_sjm_final_reconciled.csv",
]
missing = [name for name in required if not (PROJECT_ROOT / name).exists()]
if missing:
    raise FileNotFoundError(missing)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("Python", sys.version.split()[0], "| 입력 확인 완료")


## 2. 네 후보 실행

실험 코드는 성장·물가 점수를 ±0.2 히스테리시스로 분류하고, Hard 40% 이후의 월별 변동성 목표 레버리지에 각각 1.0·1.1·1.2·1.3 상한을 적용합니다. 기존 Robust VKOSPI 일간 상대수익을 마지막에 결합합니다.


In [ ]:
command = [sys.executable, "-m", "strategies.stage09_hysteresis.hysteresis_hard40_leverage_experiment"]
completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
print(completed.stdout)
if completed.returncode:
    print(completed.stderr)
    raise RuntimeError(f"실험 실패: exit={completed.returncode}")


## 3. 사전 선택표와 핵심 결론

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

results = PROJECT_ROOT / "results"
report = json.loads((results / "hysteresis_hard40_leverage_validation.json").read_text(encoding="utf-8"))
calibration = pd.read_csv(results / "hysteresis_hard40_leverage_calibration.csv")
display(calibration[[
    "Candidate", "LeverageCap", "Cal_CAGR", "Cal_Sharpe", "Cal_MDD",
    "Validation_CAGR", "Validation_Sharpe", "Validation_MDD",
    "MultiObjectiveScore", "Selected", "StrictPrelockPass",
]].style.format({
    "Cal_CAGR": "{:.2%}", "Cal_Sharpe": "{:.3f}", "Cal_MDD": "{:.2%}",
    "Validation_CAGR": "{:.2%}", "Validation_Sharpe": "{:.3f}", "Validation_MDD": "{:.2%}",
    "MultiObjectiveScore": "{:.3f}",
}))
display(Markdown(
    f"**사전 선택:** `{report['selection']['selected_candidate']}`  \n"
    f"**엄격 통과 후보:** {report['selection']['strict_eligible_count']}개  \n"
    f"**운영 판단:** `{report['selection']['promotion_status']}`"
))


## 4. 2007–2026 및 2018–2026 비교

In [ ]:
comparison = pd.read_csv(results / "hysteresis_hard40_leverage_comparison.csv")
view = comparison.loc[comparison["Period"].isin(["full_2007_2026", "locked_2018_2026"]), [
    "Period", "Strategy", "Months", "CAGR", "Volatility", "Sharpe", "MDD", "Calmar"
]]
display(view.style.format({
    "CAGR": "{:.2%}", "Volatility": "{:.2%}", "Sharpe": "{:.3f}",
    "MDD": "{:.2%}", "Calmar": "{:.3f}",
}))


## 5. 경로 차트

In [ ]:
import matplotlib.pyplot as plt

current = pd.read_csv(results / "balanced_logistic_no_sjm_final_reconciled.csv", index_col=0)
selected = pd.read_csv(results / "hysteresis_hard40_leverage_selected_reconciled.csv", index_col=0)
current.index = pd.PeriodIndex(current.index, freq="M").to_timestamp()
selected.index = pd.PeriodIndex(selected.index, freq="M").to_timestamp()

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(current.index, current["nav"], label="Current / cap 1.5", lw=2)
axes[0].plot(selected.index, selected["nav"], label="Hysteresis / cap 1.0", lw=2)
axes[0].set_title("Cumulative NAV")
axes[0].legend()
axes[0].grid(alpha=.25)
axes[1].plot(current.index, current["drawdown"] * 100, label="Current / cap 1.5", lw=2)
axes[1].plot(selected.index, selected["drawdown"] * 100, label="Hysteresis / cap 1.0", lw=2)
axes[1].set_title("Drawdown (%)")
axes[1].legend()
axes[1].grid(alpha=.25)
plt.tight_layout()
plt.show()


## 6. 결과 ZIP 내려받기

In [ ]:
output = Path("/content/hysteresis_hard40_leverage_results.zip") if IN_COLAB else PROJECT_ROOT / "hysteresis_hard40_leverage_results.zip"
members = [
    "hysteresis_hard40_leverage_validation.json",
    "hysteresis_hard40_leverage_calibration.csv",
    "hysteresis_hard40_leverage_comparison.csv",
    "hysteresis_hard40_leverage_selected_medium.csv",
    "hysteresis_hard40_leverage_selected_reconciled.csv",
    "hysteresis_hard40_signals.csv",
    "hysteresis_hard40_features.csv",
    "hysteresis_hard40_factor.csv",
]
with zipfile.ZipFile(output, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in members:
        archive.write(results / name, arcname=name)
print(output)
if IN_COLAB:
    files.download(str(output))
